# Contrast Response Demo: Spots

Split out of `7_contrast_response_demo.ipynb` (spots section only) -- see
`changes/notebook_reorg_2026-08-10.md`. Gratings and flash have their own notebooks
(`3_contrast_grating_demo.ipynb`, `5_flash_demo.ipynb`).

**Unverified -- no example spot-contrast data existed to test this section against
when it was written.** `SPOT_PROTOCOL_NAME` below is a guess
(`'manookinlab.protocols.ContrastResponseSpot'`), not a confirmed protocol name.
Run the setup cells, check `contrast_search['protocol_name'].unique()` for the real
name, and update `SPOT_PROTOCOL_NAME`/`SPOT_CONDITION_KEYS` before trusting anything
below.

Uses mean firing rate (`mean_rate`/`mean_rate_noise_sub`), not F1, unless this spot
stimulus turns out to be modulated/periodic (nonzero `temporalFrequency`), in which
case `build_trial_response_table` will also populate F1 columns automatically.

**UPDATED 2026-08-10 (item 3a):** same baseline-condition change as the grating and
flash notebooks -- `mean_rate_noise_sub` (and `f1_noise_sub`, if this stimulus turns
out to be periodic) now uses the per-cell average response from this experiment's
own `SPOT_CONDITION_KEYS[0]=0` trials as the baseline, not each trial's pre-stimulus
window. See `build_trial_response_table`'s docstring (`tuning.py`).


In [ ]:
import retinanalysis as ra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Find contrast-response datasets

Broad substring search (`'contrast'`) -- deliberately not exact-match, since spot
and flash protocol names aren't confirmed yet. Check the printed
`protocol_name`s below and fill in each section's `_PROTOCOL_NAME` variable
accordingly.

In [ ]:
contrast_search = ra.get_datasets_from_protocol_names('contrast')
print(contrast_search['protocol_name'].unique())
display(contrast_search)


## Choose an experiment (shared across all sections)

Only `exp_name` needs to be set. The white-noise/cell-typing chunk is found once
here (`ra.find_classified_noise_chunk`, same picker as the DS/OS demo -- prefers
a classified NDF0 chunk, falls back to NDF1, prints what it found) and reused by
every section below, since cell typing doesn't depend on which contrast stimulus
is being analyzed. Set `MANUAL_ANALYSIS_CHUNK` to override.

In [ ]:
exp_name = '20251222A'

MANUAL_ANALYSIS_CHUNK = None  # e.g. 'chunk13' or 'data001' -- set this to skip auto-detection

analysis_chunk_name = MANUAL_ANALYSIS_CHUNK or ra.find_classified_noise_chunk(exp_name)
if analysis_chunk_name is None:
    print(
        'No classified NDF0/NDF1 white-noise chunk found -- each section below will '
        'fall back to create_mea_pipeline\'s own nearest-chunk logic. This may crash; '
        'if so, set MANUAL_ANALYSIS_CHUNK above to a chunk you know is classified.'
    )


## Load cell typing (shared across all sections)

Builds an `AnalysisChunk` for the chunk found above just to read its
classification file(s) -- `b_load_spatial_maps=False`/`include_ei=False` since
none of this notebook needs RF maps or EIs, only the cell_id -> cell_type
mapping. If the chunk has more than one classification file, the first one found
is used by default; set `PREFERRED_TYPING_FILE` to pick a specific one.

In [ ]:
PREFERRED_TYPING_FILE = None  # e.g. 'chunk13.classificationYT.txt' -- set to pick a specific file

cell_type_map = pd.Series(dtype=object)

if analysis_chunk_name is not None:
    # NOTE (RTMP tag / "Globals file does not have RTMP tag" errors):
    # Some chunks' .globals files are missing the RTMP (runtime movie parameters) tag
    # that Vision normally writes. That tag is only used to correct for STA-vs-noise-grid
    # cropping (deltaXChecks/deltaYChecks in AnalysisChunk.get_noise_params()). When the
    # tag is missing, retinanalysis now falls back to assuming NO cropping occurred
    # (staXChecks = numXChecks, staYChecks = numYChecks, delta = 0) instead of crashing,
    # and prints a warning whenever this happens. This assumption is based on my old
    # MATLAB pipeline never needing the globals file/RTMP for this correction and never
    # having this issue -- consistent with our white-noise protocol not actually cropping
    # the STA relative to the full noise grid. If cell RF center positions ever look off
    # for a chunk that hit this warning, that assumption is the first thing to revisit.
    typing_chunk = ra.AnalysisChunk(
        exp_name, analysis_chunk_name, b_load_spatial_maps=False, include_ei=False, verbose=False,
    )
    classification_files = [f for f in typing_chunk.typing_files if 'classification' in f.lower()]
    if not classification_files:
        print(f'{analysis_chunk_name} has no classification file -- cell_type will be "Unknown" for every cell.')
    else:
        chosen_file = (
            PREFERRED_TYPING_FILE if PREFERRED_TYPING_FILE in classification_files
            else classification_files[0]
        )
        if len(classification_files) > 1:
            print(f'Multiple classification files found: {classification_files}. Using {chosen_file}.')
        file_idx = typing_chunk.typing_files.index(chosen_file)
        cell_type_map = pd.Series(
            typing_chunk.df_cell_params[f'typing_file_{file_idx}'].values,
            index=typing_chunk.cell_ids,
        )
        print(f'Loaded cell types for {len(cell_type_map)} cells from {chosen_file}.')
        print(cell_type_map.value_counts())

        # If most cells are "Unknown", the likely cause is that
        # src/retinanalysis/assets/cell_types.csv (the reference vocabulary cell-type
        # labels get matched against) is mostly a primate cell-type list, with only a few
        # mouse types patched in ('on/brisk sustained', 'on/brisk transient',
        # 'brisk sustained' -- note the '/' and the missing 'off' variants). If your
        # classification file's real labels don't exactly match an entry in that CSV,
        # matching fails silently and everything becomes "Unknown". This prints the raw
        # label text actually found in the classification file for a few cells, plus the
        # current cell_types.csv vocabulary, so the mismatch (if there is one) is visible.
        if len(cell_type_map) > 0 and (cell_type_map == 'Unknown').mean() > 0.5:
            print()
            print('Over half of cells are "Unknown" -- likely a cell_types.csv vocabulary '
                  'mismatch, not a real absence of classifications.')
            typing_file_path = typing_chunk._resolve_typing_file_path(chosen_file)
            if typing_file_path is not None:
                print(f'Raw lines from {typing_file_path}:')
                with open(typing_file_path, 'r') as f:
                    raw_lines = [line.strip() for line in f if line.strip()][:5]
                for line in raw_lines:
                    print(f'  {line!r}')
            else:
                print(f'Could not resolve a file path for {chosen_file} to show raw content.')
            import importlib.resources as ir
            import retinanalysis
            cell_types_csv_path = str(ir.files(retinanalysis) / 'assets/cell_types.csv')
            with open(cell_types_csv_path, 'r') as f:
                csv_entries = [l.strip() for l in f if l.strip() and l.strip() != 'cell_types']
            print(f'Current assets/cell_types.csv vocabulary ({len(csv_entries)} entries): {csv_entries}')
else:
    print('No analysis_chunk_name -- skipping cell typing, every cell will show as "Unknown".')


## Shared response-extraction and plotting functions

Defined in `src/retinanalysis/utils/contrast_response_utils.py` and imported below; used
by all three sections.

- `load_contrast_section(...)`: finds the datafile for a given protocol (via
  `ra.find_datafile_for_protocol`), builds the pipeline (reusing the shared
  `analysis_chunk_name` from above, with a settable `corr_cutoff` EI-matching threshold),
  builds the tidy response table (via `ra.build_trial_response_table`), tags each row with
  its `cell_type` (from `pipeline.resp.df_spike_times`, via `MEAPipeline`'s own EI-based
  cross-chunk cell mapping), and prints `epoch_parameters` keys for verification.
- `plot_crf(...)`: raw vs. noise-subtracted (rows, `show_noise_sub=False` to skip the
  noise-subtracted row) x non-normalized vs. per-cell-normalized (cols) -- population mean
  +/- SEM vs. condition.
- `plot_raster_overview_by_cell_type(...)` / `plot_rasters_for_cell_type(...)`: one figure
  PER cell type (not one combined grid) with a representative sample of cells, and every
  cell of a chosen type, paginated.

**Cell-type bookkeeping labels:** `'Unmatched'` means a cell never EI-matched the reference
classification chunk at all; `'Unknown'` means it matched but its classification label was
blank/unrecognized. Both are excluded from cell-type lists by default -- if most cells come
out "Unknown", check whether `assets/cell_types.csv` has your lab's actual mouse cell-type
vocabulary (see the typing cell above).


In [ ]:
# Shared response-extraction and plotting functions for this notebook live in
# src/retinanalysis/utils/contrast_response_utils.py, not in this cell -- this just imports
# them under their bare names, so every other cell that calls
# load_contrast_section(...)/plot_crf(...)/etc. needs no changes. Edit that file to change
# what these functions do, then re-run this cell (or restart the kernel) to pick up the
# change.
from retinanalysis.utils.contrast_response_utils import *


---
# Section: Spots

**Protocol name not confirmed from here** -- I have no example spot-contrast
data. Check `contrast_search['protocol_name'].unique()` above once you run this
notebook and set `SPOT_PROTOCOL_NAME` below to whatever the real string is (a
guess like `'manookinlab.protocols.ContrastResponseSpot'` is left as a
placeholder -- replace it). Also double check `condition_keys` below matches
this protocol's actual `epoch_parameters` (the cell below prints them for you)
-- `'contrast'` is assumed but may be named differently for this protocol.

Uses mean firing rate (`mean_rate`/`mean_rate_noise_sub`), not F1, unless this spot stimulus turns out to be
modulated/periodic (has a nonzero `temporalFrequency`), in which case
`build_trial_response_table` will also populate F1 columns automatically and you
can switch `plot_crf`'s `response_col`/`raw_response_col` to `'f1_noise_sub'`/`'f1'`
like the grating section does.

In [ ]:
SPOT_PROTOCOL_NAME = 'manookinlab.protocols.ContrastResponseSpot'  # <-- VERIFY/REPLACE from contrast_search above
SPOT_CONDITION_KEYS = ['contrast']  # <-- VERIFY against the printed epoch_parameters keys below
MANUAL_DATAFILE_NAME_SPOT = None  # e.g. 'data005' -- set to skip auto-detection
CORR_THRESHOLD_SPOT = 0.85  # EI-matching cutoff (reference chunk <-> this datafile)

df_trials_spot = spike_times_by_cell_spot = df_epochs_spot = datafile_name_spot = ndf_spot = None

if SPOT_PROTOCOL_NAME not in contrast_search['protocol_name'].unique():
    print(
        f'{SPOT_PROTOCOL_NAME!r} was not found in contrast_search -- this is a placeholder, '
        'not a confirmed protocol name. Update SPOT_PROTOCOL_NAME to one of: '
        f'{sorted(contrast_search["protocol_name"].unique())}. '
        'Skipping the rest of the Spots section until this is fixed.'
    )
else:
    # Loading prints collapse into a scrollable box.
    with scrollable_prints():
        df_trials_spot, spike_times_by_cell_spot, df_epochs_spot, datafile_name_spot, ndf_spot = load_contrast_section(
            exp_name, contrast_search, SPOT_PROTOCOL_NAME, condition_keys=SPOT_CONDITION_KEYS,
            analysis_chunk_name=analysis_chunk_name, corr_cutoff=CORR_THRESHOLD_SPOT,
            manual_datafile_name=MANUAL_DATAFILE_NAME_SPOT, typing_chunk=typing_chunk,
            baseline_condition_key=SPOT_CONDITION_KEYS[0], baseline_condition_value=0.0,
        )


## Spot response curve (mean rate) and rasters

In [ ]:
if df_trials_spot is None:
    print('Spots section not runnable yet (see the cell above) -- skipping this plot.')
else:
    # log_x=False, even_spacing=True (per yas, 2026-08-06): evenly-spaced linear axis,
    # matching the grating CRF cells above.
    fig = plot_crf(
        df_trials_spot, condition_key=SPOT_CONDITION_KEYS[0],
        response_col='mean_rate_noise_sub', raw_response_col='mean_rate',
        title=f'Spots: mean rate vs. {SPOT_CONDITION_KEYS[0]} (NDF {ndf_spot})',
        log_x=False, even_spacing=True,
    )


In [ ]:
# Log-scale version of the plot above (per yas, 2026-08-06: "for the log scale
# one its fine to have it be scientific notation") -- same data, true-to-scale
# log-x axis instead of the evenly-spaced linear one.
if df_trials_spot is None:
    print('Spots section not runnable yet (see the cell above) -- skipping this plot.')
else:
    fig = plot_crf(
        df_trials_spot, condition_key=SPOT_CONDITION_KEYS[0],
        response_col='mean_rate_noise_sub', raw_response_col='mean_rate',
        title=f'Spots: mean rate vs. {SPOT_CONDITION_KEYS[0]} (NDF {ndf_spot}) -- log scale',
        log_x=True,
    )


In [ ]:
if df_trials_spot is None:
    print('Spots section not runnable yet (see the cell above) -- skipping this plot.')
else:
    raster_figs_spot = plot_raster_overview_by_cell_type(
        df_trials_spot, spike_times_by_cell_spot, df_epochs_spot,
        condition_key=SPOT_CONDITION_KEYS[0], response_col='mean_rate_noise_sub',
    )


In [ ]:
if df_trials_spot is None:
    print('Spots section not runnable yet (see the cell above) -- skipping this plot.')
else:
    # Excludes 'Unknown'/'Unmatched' bookkeeping labels so this defaults to a real,
    # CSV-matching cell type.
    _real_types_spot = sorted(
        t for t in df_trials_spot['cell_type'].unique() if t not in ('Unknown', 'Unmatched')
    )
    SELECTED_CELL_TYPE_SPOT = (
        _real_types_spot[0] if _real_types_spot
        else sorted(df_trials_spot['cell_type'].unique())[0]
    )  # change to any type printed above
    print(f'Selected cell type: {SELECTED_CELL_TYPE_SPOT!r} (available: {sorted(df_trials_spot["cell_type"].unique())})')

    raster_figs_spot_selected = plot_rasters_for_cell_type(
        df_trials_spot, spike_times_by_cell_spot, df_epochs_spot,
        condition_key=SPOT_CONDITION_KEYS[0], selected_cell_type=SELECTED_CELL_TYPE_SPOT,
    )


### Optional: look at a different NDF (light level) or cell type

Same idea as the grating section's explorer -- reruns this section's rasters at a
different light level via `ra.get_ndf_blocks_for_protocol`. Leave
`EXPLORE_NDF_SPOT = None` to skip this. Only runs if the Spots section above found
real data (SPOT_PROTOCOL_NAME confirmed).

In [ ]:
EXPLORE_NDF_SPOT = None  # e.g. 2.0 -- set to an NDF value to look at a different light level
EXPLORE_CELL_TYPE_SPOT = None  # e.g. 'off brisk sustained' -- None reuses SELECTED_CELL_TYPE_SPOT

if df_trials_spot is None:
    print('Spots section not runnable yet -- skipping the NDF explorer too.')
elif EXPLORE_NDF_SPOT is not None:
    df_ndf_blocks_spot = ra.get_ndf_blocks_for_protocol(exp_name, SPOT_PROTOCOL_NAME)
    match = df_ndf_blocks_spot[df_ndf_blocks_spot['NDF'] == EXPLORE_NDF_SPOT]
    if len(match) == 0:
        print(f'No {SPOT_PROTOCOL_NAME} block found at NDF {EXPLORE_NDF_SPOT} for {exp_name}. '
              f'Available NDFs: {list(df_ndf_blocks_spot["NDF"])}')
    else:
        explore_datafile = match.iloc[0]['datafile_name']
        print(f'Loading NDF {EXPLORE_NDF_SPOT} ({explore_datafile}) ...')
        # Loading prints collapse into a scrollable box; the raster figures below
        # render normally.
        with scrollable_prints():
            df_trials_explore, spike_times_by_cell_explore, df_epochs_explore, _, _ = load_contrast_section(
                exp_name, contrast_search, SPOT_PROTOCOL_NAME, condition_keys=SPOT_CONDITION_KEYS,
                analysis_chunk_name=analysis_chunk_name, corr_cutoff=CORR_THRESHOLD_SPOT,
                manual_datafile_name=explore_datafile,
                baseline_condition_key=SPOT_CONDITION_KEYS[0], baseline_condition_value=0.0,
            )
        explore_cell_type = EXPLORE_CELL_TYPE_SPOT or SELECTED_CELL_TYPE_SPOT
        raster_figs_explore_spot = plot_rasters_for_cell_type(
            df_trials_explore, spike_times_by_cell_explore, df_epochs_explore,
            condition_key=SPOT_CONDITION_KEYS[0], selected_cell_type=explore_cell_type,
        )
else:
    print('EXPLORE_NDF_SPOT is None -- skipping. Set it to an NDF value to explore a different light level.')
